In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from combra import data
from combra.metrics import wdist_vs_n

types_dict = {
    'Ultra_Co11': 'medium grain',
    'Ultra_Co25': 'small grain',
    'Ultra_Co8':  'medium-small grain',
    'Ultra_Co6_2': 'large grain',
    'Ultra_Co15': 'medium-small grain',
}

# H5 sources for the diffit kimg checkpoints. Update this list as new kimg dumps appear.
h5_paths = [
    './data/generated_images_h5/00017-diffit-256-gpus2-batch192_kimg_004435.h5',
    './data/generated_images_h5/00017-diffit-256-gpus2-batch192_kimg_006451.h5',
    './data/generated_images_h5/00017-diffit-256-gpus2-batch192_kimg_008064.h5',
    './data/generated_images_h5/00017-diffit-256-gpus2-batch192_kimg_012096.h5',
    './data/generated_images_h5/00017-diffit-256-gpus2-batch192_kimg_016128.h5',
]

# Ethalon (real) parquet for W-dist comparisons
ethalon_path = './data/angles/o_bc_left_4x_768_256x256_N360_msl5/angles_n1000.parquet'

# diffit was trained on 3 classes, so its outputs are class_0/1/2; ethalon uses class_Ultra_Co*.
class_map = {
    'class_0': 'class_Ultra_Co11',
    'class_1': 'class_Ultra_Co25',
    'class_2': 'class_Ultra_Co6_2',
}

# Generate angle parquets at full N

One parquet per kimg checkpoint, all step values in a single file. Output layout
mirrors the existing `./data/angles/<run>_msl5/angles_n{N}.parquet` convention so
that `combra.metrics.compare_folders` and the rest of the analysis notebooks pick
the files up automatically.

In [ ]:
out_root = Path('./angles')

for h5 in h5_paths:
    save_path = out_root / (Path(h5).stem + '_msl5')
    print(f'\n=== {h5} -> {save_path} ===')
    ds = data.PobeditDataset(path=h5, max_images_num_per_class=None)  # full N
    ds.generate_angles(
        save_path=str(save_path),
        types_dict=types_dict,
        step=[0.1, 0.5, 1, 2, 3, 4, 5],
        workers=20,
        angles_tol=3,
        min_segment_len=5.0,
        keep_contours=False,
        chunksize=64,
    )

# N-sweep generation for the W-dist convergence study

For each H5 source we re-run angle extraction at several truncated sample sizes,
producing one parquet per N value. PobeditDataset's filename convention writes
each run as `angles_n{N}.parquet`, so they all coexist in one per-generator
folder under `./angles/<run>_nsweep/`.

Note: the prep cache is keyed by N — each N triggers a fresh preprocess pass.
For full sweeps over all five kimg dumps this is a multi-hour job. Start with a
short `n_values` list to validate, then expand.

In [ ]:
n_values = [100, 250, 500, 1000]
sweep_root = Path('./angles_nsweep')

for h5 in h5_paths:
    sweep_dir = sweep_root / Path(h5).stem
    sweep_dir.mkdir(parents=True, exist_ok=True)
    print(f'\n=== sweep {h5} -> {sweep_dir} ===')
    for N in n_values:
        print(f'  N={N}')
        ds = data.PobeditDataset(path=h5, max_images_num_per_class=N)
        ds.generate_angles(
            save_path=str(sweep_dir),
            types_dict=types_dict,
            step=[5.0],  # convergence study uses the 5-deg quantization
            workers=20,
            angles_tol=3,
            min_segment_len=5.0,
            keep_contours=False,
            chunksize=64,
        )

# W-dist vs N convergence plot

For each generator (one per kimg checkpoint), W-dist between its angle distribution
and the ethalon is plotted against N on log-x. The dashed reference line is
`1 / (100·sqrt(N))` — the expected scaling for empirical-distribution Wasserstein
distance under iid sampling.

Cells above must have produced the parquets under `./angles_nsweep/<run>/` first.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

sweep_root = Path('./angles_nsweep')

for h5 in h5_paths:
    sweep_dir = sweep_root / Path(h5).stem
    if not sweep_dir.exists():
        print(f'[skip] {sweep_dir} missing')
        continue

    records = wdist_vs_n(sweep_dir, ethalon_path, class_map=class_map, step=5.0)
    if not records:
        print(f'[skip] no records for {sweep_dir}')
        continue

    # Average W-dist across classes per N value to get one curve per generator.
    by_n = {}
    for r in records:
        by_n.setdefault(r['n_images'], []).append(r['w_dist'])
    Ns = sorted(by_n)
    ws = [np.mean(by_n[n]) for n in Ns]
    label = sweep_dir.name.split('_kimg_')[-1] if '_kimg_' in sweep_dir.name else sweep_dir.name
    ax.plot(Ns, ws, '-o', label=f'kimg {label}')

# Reference: empirical-distribution Wasserstein scales like 1/sqrt(N).
ref_N = np.array([100, 200, 500, 1000, 5000, 10000])
ax.plot(ref_N, 1 / (100 * np.sqrt(ref_N)), '--k', label=r'$1/(100\sqrt{N})$')

ax.set_xscale('log')
ax.set_xlabel('Number of images, N')
ax.set_ylabel('Wasserstein distance (mean over classes)')
ax.set_title('Angle-distribution convergence vs sample size')
ax.legend()
plt.show()

# Per-class convergence (one subplot per grain class)

Same data as the previous plot, broken out by class. Each generator's parquets
are loaded once and reused across all subplots.

In [ ]:
classes = ['class_Ultra_Co11', 'class_Ultra_Co25', 'class_Ultra_Co6_2']

# Load each generator's records once, then dispatch into per-class subplots.
records_by_gen = {}
for h5 in h5_paths:
    sweep_dir = sweep_root / Path(h5).stem
    if not sweep_dir.exists():
        continue
    records_by_gen[h5] = wdist_vs_n(sweep_dir, ethalon_path, class_map=class_map, step=5.0)

if not records_by_gen:
    print('No nsweep parquets found under', sweep_root.resolve())
else:
    fig, axes = plt.subplots(len(classes), 1, figsize=(10, 4 * len(classes)),
                             gridspec_kw={'hspace': 0.4}, squeeze=False)

    for ax, cls in zip(axes[:, 0], classes):
        for h5, recs in records_by_gen.items():
            pts = sorted((r['n_images'], r['w_dist']) for r in recs if r['class'] == cls)
            if not pts:
                continue
            Ns, ws = zip(*pts)
            stem = Path(h5).stem
            label = stem.split('_kimg_')[-1] if '_kimg_' in stem else stem
            ax.plot(Ns, ws, '-o', label=f'kimg {label}')

        ref_N = np.array([100, 200, 500, 1000, 5000, 10000])
        ax.plot(ref_N, 1 / (100 * np.sqrt(ref_N)), '--k', label=r'$1/(100\sqrt{N})$')
        ax.set_xscale('log')
        ax.set_xlabel('Number of images, N')
        ax.set_ylabel('Wasserstein distance')
        ax.set_title(cls)
        ax.legend(fontsize=8)

    plt.show()